In [0]:
%pip install pytest

In [0]:
%run "../src/utils/02_cleaning_transformation_to_silver_utils"

In [0]:
import pytest
import sys
from pyspark.sql.functions import col
from pyspark.testing.utils import assertDataFrameEqual

In [0]:
def test_bronze_sql_customers_silver():
    raw_data = [
        ("C10139  ", "  bryan  ", "  wiggins  ", "  MARYstone@example.com  ", "(495)657-9887x74924", 
         "5402 Mitchell NW Apt. 3B", "lake amanda", "  latam  ", "2023-09-10", "smb", None, 
         "2026-07-17T20:08:21.026Z", "/Volumes/novamart/bronze/customers/file.csv")
    ]
    raw_schema = [
        "customer_id", "first_name", "last_name", "email", "phone", 
        "address", "city", "region", "signup_date", "customer_segment", 
        "_rescued_data", "ingesttime", "file_name"
    ]

    df_raw_mock = spark.createDataFrame(raw_data, raw_schema)
    df_actual = bronze_sql_customers_silver(df_raw_mock)

    expected_schema = [
        "customer_id", "first_name", "last_name", "email", "address", 
        "city", "region", "customer_segment", "full_name", "verified_email", 
        "cleaned_phone", "verified_phone"
    ]
    expected_data = [
        (
            "C10139",                     # 0. customer_id (clean)
            "Bryan",                      # 1. first_name (capitalize)
            "Wiggins",                    # 2. last_name (capitalize)
            "marystone@example.com",      # 3. email (lowercase, no spaces!)
            "5402 Mitchell NW Apt. 3B",   # 4. address (spaces clean, case safe)
            "Lake Amanda",                # 5. city (capitalize)
            "LATAM",                      # 6. region (acronym capital)
            "Smb",                        # 7. customer_segment (capitalize)
            "Bryan Wiggins",              # 8. full_name (concatenated)
            True,                         # 9. verified_email (boolean)
            "+14956579887x74924",         # 10. cleaned_phone (E164 standard with extension!)
            True                          # 11. verified_phone (boolean)
        )
    ]

    df_expected = spark.createDataFrame(expected_data, expected_schema)

    assertDataFrameEqual(
        df_actual.select(*expected_schema), df_expected
    )